In [0]:
# from delta.tables import DeltaTable
# from pyspark.sql import functions as F

# # Exact identity of the October 9 customer file
# target_path = (
#     base_path
#     + "landing/customers/year=2026/month=10/day=09/"
#     + "customers_20261009.json"
# )

# old_run_id = "FILE_MANIFEST_20261010062759_a471b98b"

# # Confirm that exactly the expected record is stuck
# target_condition = (
#     (F.col("dataset") == "customers")
#     & (F.col("file_path") == target_path)
#     & (F.col("status") == "PROCESSING")
#     & (F.col("run_id") == old_run_id)
# )

# manifest_df = (
#     spark.read
#     .format("delta")
#     .load(manifest_path)
# )

# matching_count = manifest_df.filter(target_condition).count()

# if matching_count != 1:
#     raise RuntimeError(
#         f"Expected exactly 1 matching October 9 record, "
#         f"but found {matching_count}. No changes made."
#     )

# # Release this exact claim for a new incremental run
# manifest_table = DeltaTable.forPath(spark, manifest_path)

# manifest_table.update(
#     condition=target_condition,
#     set={
#         "status": F.lit("NEW"),
#         "processed_at": F.lit(None).cast("timestamp"),
#         "error_message": F.lit(None).cast("string")
#     }
# )

# # Verify the updated record
# display(
#     spark.read
#     .format("delta")
#     .load(manifest_path)
#     .filter(
#         (F.col("dataset") == "customers")
#         & (F.col("file_path") == target_path)
#     )
#     .select(
#         "file_name",
#         "status",
#         "run_id",
#         "processed_at",
#         "error_message"
#     )
# )

file_name,status,run_id,processed_at,error_message
customers_20261009.json,NEW,FILE_MANIFEST_20261010062759_a471b98b,null,null


In [0]:
# ============================================================
# CELL 1 — IMPORTS, WIDGETS, AND CONFIGURATION
# ============================================================

from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)
from delta.tables import DeltaTable

In [0]:
def ensure_text_widget(name, default_value, label):
    """Create a text widget only if it does not already exist."""
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default_value, label)


def ensure_dropdown_widget(name, default_value, choices, label):
    """Create a dropdown widget only if it does not already exist."""
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default_value, choices, label)


# This must be the exact workspace path of file_manifest_v2.
# Fill it in from the widget after running this cell.
ensure_text_widget(
    "manifest_notebook_path",
    "",
    "Manifest Notebook Path"
)

ensure_dropdown_widget(
    "manifest_run_mode",
    "incremental",
    ["incremental", "retry"],
    "Manifest Run Mode"
)

In [0]:


# Project paths: keep these aligned with the existing project.
base_path = (
    "abfss://customer-sales-datalake@"
    "nestlesalesdatalake.dfs.core.windows.net/"
)
landing_customers_root = base_path + "landing/customers/"
bronze_path = base_path + "bronze/customers/"
manifest_path = base_path + "metadata/file_manifest/"

DATASET_NAME = "customers"
SOURCE_SYSTEM = "customer_api"
REQUIRED_SOURCE_COLUMNS = ["customer_id", "name", "city", "market"]
BRONZE_COLUMNS = [
    "customer_id",
    "name",
    "city",
    "market",
    "ingestion_timestamp",
    "source_system",
    "source_file",
    "run_id",
    "batch_id"
]

manifest_notebook_path = (
    dbutils.widgets.get("manifest_notebook_path").strip()
)
manifest_run_mode = (
    dbutils.widgets.get("manifest_run_mode").strip().lower()
)

print("Dataset:", DATASET_NAME)
print("Manifest mode:", manifest_run_mode)
print("Landing customer root:", landing_customers_root)
print("Bronze destination:", bronze_path)
print("Manifest path:", manifest_path)

if not manifest_notebook_path:
    print(
        "ACTION REQUIRED: enter the full workspace path of "
        "file_manifest_v2 in the Manifest Notebook Path widget."
    )

Dataset: customers
Manifest mode: retry
Landing customer root: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/
Bronze destination: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/customers/
Manifest path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/file_manifest/


In [0]:
# ============================================================
# CELL 2 — EXECUTE FILE MANIFEST FOR CUSTOMERS ONLY
# ============================================================

if not manifest_notebook_path:
    raise ValueError(
        "Enter the exact workspace path of file_manifest "
        "in the Manifest Notebook Path widget, then rerun this cell."
    )

if not manifest_notebook_path.startswith("/"):
    raise ValueError(
        "Use an absolute Databricks workspace path for the manifest notebook."
    )

if manifest_run_mode not in {"incremental", "retry"}:
    raise ValueError(
        "This notebook currently supports incremental and retry modes only. "
        "A true full rebuild needs a separate, explicit rebuild strategy."
    )

In [0]:


# The called manifest notebook must have these widgets:
# run_mode, start_date, end_date, dataset_filter
# It must finish with: dbutils.notebook.exit(run_id)
manifest_arguments = {
    "run_mode": manifest_run_mode,
    "start_date": "",
    "end_date": "",
    "dataset_filter": DATASET_NAME
}

print("Executing manifest notebook:", manifest_notebook_path)
print("Manifest arguments:", manifest_arguments)

manifest_run_id = dbutils.notebook.run(
    manifest_notebook_path,
    3600,
    manifest_arguments
).strip()

if not manifest_run_id:
    raise RuntimeError(
        "The manifest notebook returned an empty run ID. "
        "Confirm its final statement is dbutils.notebook.exit(run_id)."
    )

print("Manifest execution completed.")
print("Manifest run ID:", manifest_run_id)

Executing manifest notebook: /Workspace/Users/prachethanreddy01@gmail.com/customer-sales-project/00_setup/file_manifest
Manifest arguments: {'run_mode': 'retry', 'start_date': '', 'end_date': '', 'dataset_filter': 'customers'}
Manifest execution completed.
Manifest run ID: FILE_MANIFEST_20261010161207_0d3d6785


In [0]:
# ============================================================
# CELL 3 — LOAD FILES CLAIMED BY THIS MANIFEST RUN
# ============================================================

if not DeltaTable.isDeltaTable(spark, manifest_path):
    raise RuntimeError(f"Manifest Delta table not found: {manifest_path}")

manifest_df = (
    spark.read
    .format("delta")
    .load(manifest_path)
)

claimed_files_df = (
    manifest_df
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("status") == "PROCESSING")
        & (F.col("run_id") == manifest_run_id)
    )
    .select(
        "dataset",
        "file_name",
        "file_path",
        "batch_id",
        "run_id",
        "retry_count"
    )
)

# Duplicate manifest identities should be fixed, not silently processed twice.
duplicate_claims_df = (
    claimed_files_df
    .groupBy("dataset", "file_path")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_claims_df.count() > 0:
    display(duplicate_claims_df)
    raise RuntimeError(
        "Duplicate customer file identities were claimed by this run. "
        "Resolve the duplicate manifest records first."
    )

claimed_count = claimed_files_df.count()
print("Customer files claimed by this run:", claimed_count)
display(claimed_files_df.orderBy("file_path"))

Customer files claimed by this run: 1


dataset,file_name,file_path,batch_id,run_id,retry_count
customers,customers_20261009.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=09/customers_20261009.json,23ff03ec-4441-456b-ace1-f4f3c0fa49db,FILE_MANIFEST_20261010161207_0d3d6785,1


In [0]:
# ============================================================
# CELL 4 — HELPER TO UPDATE MANIFEST STATUS SAFELY
# ============================================================

status_update_schema = StructType([
    StructField("dataset", StringType(), False),
    StructField("file_path", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("new_status", StringType(), False),
    StructField("error_message", StringType(), True)
])


def update_manifest_status(dataset, file_path, run_id, new_status, error_message=None):
    """Update a claimed manifest row only if this run still owns it."""

    if new_status not in {"SUCCESS", "FAILED"}:
        raise ValueError(f"Unsupported final status: {new_status}")

    manifest_table = DeltaTable.forPath(spark, manifest_path)

    manifest_table.update(
        condition=(
            (F.col("dataset") == F.lit(dataset))
            & (F.col("file_path") == F.lit(file_path))
            & (F.col("status") == F.lit("PROCESSING"))
            & (F.col("run_id") == F.lit(run_id))
        ),
        set={
            "status": F.lit(new_status),
            "processed_at": F.current_timestamp(),
            "error_message": F.lit(error_message).cast("string")
        }
    )

    # Verify that the intended final state is persisted.
    latest_df = (
        spark.read
        .format("delta")
        .load(manifest_path)
    )

    final_count = (
        latest_df
        .filter(
            (F.col("dataset") == dataset)
            & (F.col("file_path") == file_path)
            & (F.col("run_id") == run_id)
            & (F.col("status") == new_status)
        )
        .count()
    )

    if final_count != 1:
        raise RuntimeError(
            f"Could not verify manifest status {new_status} for {file_path}. "
            "The file may no longer be owned by this run."
        )

In [0]:
# ============================================================
# CELL 5 — INGEST CLAIMED CUSTOMER FILES INTO BRONZE
# ============================================================

# The Bronze layer preserves the source attributes. Cleaning, validation,
# deduplication, quarantine, and SCD Type 2 logic belong in Silver.

processed_files = 0
skipped_existing_files = 0
failed_files = []

claimed_file_records = claimed_files_df.collect()

if not claimed_file_records:
    print(
        "No customers files were claimed by this manifest run. "
        "No Bronze writes were performed."
    )

for file_record in claimed_file_records:

    dataset = file_record["dataset"]
    file_name = file_record["file_name"]
    file_path = file_record["file_path"]
    batch_id = file_record["batch_id"]

    print("\n============================================")
    print("Processing customer file:", file_name)
    print("Source path:", file_path)
    print("Batch ID:", batch_id)
    print("Manifest run ID:", manifest_run_id)
    print("============================================")

    try:
        # Confirm the file exists and can be read as a multiline JSON array.
        source_df = (
            spark.read
            .option("multiLine", "true")
            .json(file_path)
        )

        missing_columns = [
            column
            for column in REQUIRED_SOURCE_COLUMNS
            if column not in source_df.columns
        ]

        if missing_columns:
            raise ValueError(
                f"Source file is missing required columns: {missing_columns}. "
                f"Available columns: {source_df.columns}"
            )

        source_count = source_df.count()
        print("Source records:", source_count)

        # Idempotency guard: if this exact Landing path is already in Bronze,
        # do not append the same file a second time.
        already_in_bronze = False

        if DeltaTable.isDeltaTable(spark, bronze_path):
            existing_bronze_df = (
                spark.read
                .format("delta")
                .load(bronze_path)
            )

            missing_bronze_columns = [
                column
                for column in BRONZE_COLUMNS
                if column not in existing_bronze_df.columns
            ]

            extra_bronze_columns = [
                column
                for column in existing_bronze_df.columns
                if column not in BRONZE_COLUMNS
            ]

            if missing_bronze_columns or extra_bronze_columns:
                raise ValueError(
                    "Existing Bronze schema does not match the expected "
                    f"schema. Missing={missing_bronze_columns}; "
                    f"extra={extra_bronze_columns}. Inspect the table before "
                    "changing its schema."
                )

            already_in_bronze = (
                existing_bronze_df
                .filter(F.col("source_file") == F.lit(file_path))
                .limit(1)
                .count() > 0
            )

        if already_in_bronze:
            print(
                "This exact source file is already present in Bronze. "
                "Skipping the append to avoid duplicate ingestion."
            )
            skipped_existing_files += 1

        else:
            bronze_file_df = (
                source_df
                .withColumn("customer_id", F.col("customer_id").cast("int"))
                .withColumn("name", F.col("name").cast("string"))
                .withColumn("city", F.col("city").cast("string"))
                .withColumn("market", F.col("market").cast("string"))
                .withColumn("ingestion_timestamp", F.current_timestamp())
                .withColumn("source_system", F.lit(SOURCE_SYSTEM))
                # Store the full ADLS path to identify this source file uniquely.
                .withColumn("source_file", F.lit(file_path))
                .withColumn("run_id", F.lit(manifest_run_id))
                .withColumn("batch_id", F.lit(batch_id))
                .select(*BRONZE_COLUMNS)
            )

            # Append only. If the path has no Delta table yet, this creates it.
            (
                bronze_file_df.write
                .format("delta")
                .mode("append")
                .save(bronze_path)
            )

            print("Bronze write completed for records:", source_count)

        # Mark SUCCESS only after the write (or idempotent verification) passed.
        update_manifest_status(
            dataset=dataset,
            file_path=file_path,
            run_id=manifest_run_id,
            new_status="SUCCESS",
            error_message=None
        )

        print("Manifest status updated to SUCCESS.")
        processed_files += 1

    except Exception as exc:
        error_text = f"{type(exc).__name__}: {str(exc)}"[:4000]
        print("ERROR processing file:", error_text)

        try:
            update_manifest_status(
                dataset=dataset,
                file_path=file_path,
                run_id=manifest_run_id,
                new_status="FAILED",
                error_message=error_text
            )
            print("Manifest status updated to FAILED.")
        except Exception as status_exc:
            print(
                "CRITICAL: Could not update the manifest failure status: ",
                str(status_exc)[:1000]
            )

        failed_files.append({
            "file_name": file_name,
            "file_path": file_path,
            "error_message": error_text
        })

print("\nCustomers files successfully handled:", processed_files)
print("Already-present files skipped from append:", skipped_existing_files)
print("Files with errors:", len(failed_files))


Processing customer file: customers_20261009.json
Source path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=09/customers_20261009.json
Batch ID: 23ff03ec-4441-456b-ace1-f4f3c0fa49db
Manifest run ID: FILE_MANIFEST_20261010161207_0d3d6785
Source records: 25000
Bronze write completed for records: 25000
Manifest status updated to SUCCESS.

Customers files successfully handled: 1
Already-present files skipped from append: 0
Files with errors: 0


In [0]:
# ============================================================
# CELL 6 — VERIFY FINAL MANIFEST STATUS AND BRONZE COUNTS
# ============================================================

final_manifest_df = (
    spark.read
    .format("delta")
    .load(manifest_path)
)

print("Final manifest records for this customers run:")
display(
    final_manifest_df
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("run_id") == manifest_run_id)
    )
    .select(
        "dataset",
        "file_name",
        "status",
        "processed_at",
        "run_id",
        "retry_count",
        "error_message"
    )
    .orderBy("file_name")
)

if DeltaTable.isDeltaTable(spark, bronze_path):
    final_bronze_df = (
        spark.read
        .format("delta")
        .load(bronze_path)
    )

    print("Total customer Bronze rows:", final_bronze_df.count())
    print("Bronze rows by source file:")
    display(
        final_bronze_df
        .groupBy("source_file", "batch_id")
        .count()
        .orderBy("source_file")
    )
else:
    print("Customer Bronze Delta table does not exist yet.")

if failed_files:
    raise RuntimeError(
        f"Customer Bronze ingestion failed for {len(failed_files)} file(s). "
        "Inspect the error messages and manifest error_message column."
    )

print("Customer Bronze notebook finished.")

Final manifest records for this customers run:


dataset,file_name,status,processed_at,run_id,retry_count,error_message
customers,customers_20261009.json,SUCCESS,2026-10-10T16:13:50.252Z,FILE_MANIFEST_20261010161207_0d3d6785,1,null


Total customer Bronze rows: 50000
Bronze rows by source file:


source_file,batch_id,count
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,BATCH_202610,25000
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=09/customers_20261009.json,23ff03ec-4441-456b-ace1-f4f3c0fa49db,25000


Customer Bronze notebook finished.
